In [17]:
# =========================================
# 1. IMPORTS
# =========================================
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LassoCV, ElasticNetCV

In [3]:
# 2. LOAD DATA
# =========================================
train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

In [4]:
# 3. Preprossesing (Gộp 2 hàm drop cột nhiều null + nhieeuf zero)
def drop_null_and_zero(train_df, test_df, null_threshold=500, zero_ratio=0.5):
    # 3.1 drop cột quá nhiều null trong train
    cols_null = []
    for col in train_df.columns:
        if train_df[col].isnull().sum() > null_threshold:
            cols_null.append(col)
    train_df = train_df.drop(columns=cols_null)
    # test cũng drop cùng cột
    test_df = test_df.drop(columns=[c for c in cols_null if c in test_df.columns], errors="ignore")

    # 3.2 drop cột có > zero_ratio là 0
    def _drop_zero_cols(df, cols_to_check):
        n_rows = len(df)
        cols_drop = []
        for c in cols_to_check:
            zero_count = (df[c] == 0).sum()
            if zero_count / n_rows > zero_ratio:
                cols_drop.append(c)
        return cols_drop

    # chỉ check trên train, sau đó drop cả 2
    numeric_cols = train_df.select_dtypes(include=[np.number]).columns
    zero_cols = _drop_zero_cols(train_df, numeric_cols)

    train_df = train_df.drop(columns=zero_cols)
    test_df = test_df.drop(columns=[c for c in zero_cols if c in test_df.columns], errors="ignore")

    return train_df, test_df

train_clean, test_clean = drop_null_and_zero(train, test, null_threshold=500, zero_ratio=0.5)

In [5]:
# =========================================
# 4. FEATURE ENGINEERING
# =========================================

def prepare_features(train_df, test_df, target_col="SalePrice"):
    train_df = train_df.copy()
    test_df = test_df.copy()

    # ---- chọn các cột gốc cần dùng (khoảng 33 cột + vài cột FE)
    base_cols = [
        'MSZoning','LotArea','LotConfig','LandSlope','Neighborhood',
        'HouseStyle','OverallQual','YearBuilt','YearRemodAdd',
        'RoofStyle','Exterior1st','Exterior2nd','ExterQual','Foundation',
        'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','TotalBsmtSF',
        'HeatingQC','CentralAir','1stFlrSF','GrLivArea','FullBath',
        'TotRmsAbvGrd','GarageType','GarageFinish','GarageCars','GarageArea',
        'PavedDrive','MoSold','YrSold','SaleType','SaleCondition'
    ]

    # đôi khi 1 số cột sau khi drop ở trên sẽ mất → chỉ giữ cột còn tồn tại
    base_cols = [c for c in base_cols if c in train_df.columns]

    # lấy train_X
    train_X = train_df[base_cols].copy()
    # lấy test_X
    test_X = test_df[[c for c in base_cols if c in test_df.columns]].copy()

    # =============================
    # 4.1 MAP / RÚT GỌN CÁC CỘT CATEGORICAL
    # =============================

    # BsmtFinType1
    col_BsmtFinType1 = {'Unf': 1, 'GLQ': 2, 'ALQ': 3, 'BLQ': 3, 'Rec': 3, 'LwQ': 3}
    for df in [train_X, test_X]:
        if 'BsmtFinType1' in df.columns:
            df['BsmtFinType1'] = df['BsmtFinType1'].map(col_BsmtFinType1).fillna(4)

    # BsmtExposure
    col_BsmtExposure = {'No': 1, 'Av': 2, 'Mn': 2, 'Gd': 3}
    for df in [train_X, test_X]:
        if 'BsmtExposure' in df.columns:
            df['BsmtExposure'] = df['BsmtExposure'].map(col_BsmtExposure).fillna(4)

    # BsmtQual
    col_BsmtQual = {'TA':1, 'Fa':1, 'Gd':2, 'Ex':3}
    for df in [train_X, test_X]:
        if 'BsmtQual' in df.columns:
            df['BsmtQual'] = df['BsmtQual'].map(col_BsmtQual)

    # Exterior1st
    for df in [train_X, test_X]:
        if 'Exterior1st' in df.columns:
            df['Exterior1st'] = df['Exterior1st'].apply(
                lambda x: x if x in ['VinylSd','MetalSd','Wd Sdng','HdBoard','Plywood','Stucco'] else 'others'
            )
            map_ex1 = {'VinylSd':1, 'MetalSd':2, 'Wd Sdng':2, 'HdBoard':2, 'Plywood':2, 'Stucco':2, 'others':3}
            df['Exterior1st'] = df['Exterior1st'].map(map_ex1)

    # Exterior2nd
    for df in [train_X, test_X]:
        if 'Exterior2nd' in df.columns:
            df['Exterior2nd'] = df['Exterior2nd'].apply(
                lambda x: x if x in ['VinylSd','MetalSd','HdBoard','Wd Sdng','Plywood'] else 'others'
            )
            map_ex2 = {'VinylSd':1, 'MetalSd':2, 'HdBoard':3, 'Wd Sdng':4, 'Plywood':5, 'others':6}
            df['Exterior2nd'] = df['Exterior2nd'].map(map_ex2)

    # ExterQual
    for df in [train_X, test_X]:
        if 'ExterQual' in df.columns:
            df['ExterQual'] = df['ExterQual'].apply(lambda x: x if x in ['TA','Gd'] else 'others')
            map_exq = {'TA':1, 'Gd':2, 'others':3}
            df['ExterQual'] = df['ExterQual'].map(map_exq)

    # Foundation
    for df in [train_X, test_X]:
        if 'Foundation' in df.columns:
            df['Foundation'] = df['Foundation'].apply(lambda x: x if x in ['PConc','CBlock','BrkTil'] else 'others')
            map_f = {'PConc':1, 'CBlock':2, 'BrkTil':3, 'others':4}
            df['Foundation'] = df['Foundation'].map(map_f)

    # GarageFinish
    for df in [train_X, test_X]:
        if 'GarageFinish' in df.columns:
            df['GarageFinish'] = df['GarageFinish'].fillna('others')
            map_gf = {'Unf':1, 'RFn':2, 'Fin':3, 'others':4}
            df['GarageFinish'] = df['GarageFinish'].map(map_gf)

    # GarageType
    for df in [train_X, test_X]:
        if 'GarageType' in df.columns:
            df['GarageType'] = df['GarageType'].apply(lambda x: x if x in ['Attchd','Detchd'] else 'others')
            map_gt = {'Attchd':1, 'Detchd':2, 'others':3}
            df['GarageType'] = df['GarageType'].map(map_gt)

    # HeatingQC
    for df in [train_X, test_X]:
        if 'HeatingQC' in df.columns:
            df['HeatingQC'] = df['HeatingQC'].apply(lambda x: x if x in ['Ex','TA','Gd'] else 'others')
            map_hq = {'Ex':1, 'TA':2, 'Gd':3, 'others':4}
            df['HeatingQC'] = df['HeatingQC'].map(map_hq)

    # HouseStyle
    for df in [train_X, test_X]:
        if 'HouseStyle' in df.columns:
            df['HouseStyle'] = df['HouseStyle'].apply(lambda x: x if x in ['1Story','2Story','1.5Fin'] else 'others')
            map_hs = {'1Story':1, '2Story':2, '1.5Fin':3, 'others':4}
            df['HouseStyle'] = df['HouseStyle'].map(map_hs)

    # KitchenQual nếu có
    for df in [train_X, test_X]:
        if 'KitchenQual' in df.columns:
            df['KitchenQual'] = df['KitchenQual'].apply(lambda x: x if x in ['TA','Gd','Ex'] else 'others')
            map_kq = {'TA':1, 'Gd':2, 'Ex':3, 'others':4}
            df['KitchenQual'] = df['KitchenQual'].map(map_kq)

    # LandSlope: bạn drop vì lệch → drop luôn
    for df in [train_X, test_X]:
        if 'LandSlope' in df.columns:
            df.drop(columns=['LandSlope'], inplace=True)

    # LotConfig
    for df in [train_X, test_X]:
        if 'LotConfig' in df.columns:
            df['LotConfig'] = df['LotConfig'].apply(lambda x: x if x in ['Inside','Corner'] else 'others')
            map_lc = {'Inside':1, 'Corner':2, 'others':3}
            df['LotConfig'] = df['LotConfig'].map(map_lc)

    # LotShape
    for df in [train_X, test_X]:
        if 'LotShape' in df.columns:
            df['LotShape'] = df['LotShape'].apply(lambda x: x if x in ['Reg','IR1'] else 'others')
            map_ls = {'Reg':1, 'IR1':2, 'others':3}
            df['LotShape'] = df['LotShape'].map(map_ls)

    # MSZoning
    for df in [train_X, test_X]:
        if 'MSZoning' in df.columns:
            df['MSZoning'] = df['MSZoning'].apply(lambda x: x if x in ['RL','RM'] else 'others')
            map_ms = {'RL':1, 'RM':2, 'others':3}
            df['MSZoning'] = df['MSZoning'].map(map_ms)

    # RoofStyle
    for df in [train_X, test_X]:
        if 'RoofStyle' in df.columns:
            df['RoofStyle'] = df['RoofStyle'].apply(lambda x: x if x in ['Gable','Hip'] else 'others')
            map_rs = {'Gable':1, 'Hip':2, 'others':3}
            df['RoofStyle'] = df['RoofStyle'].map(map_rs)

    # SaleCondition
    for df in [train_X, test_X]:
        if 'SaleCondition' in df.columns:
            df['SaleCondition'] = df['SaleCondition'].apply(
                lambda x: x if x in ['Normal','Partial','Abnorml'] else 'others'
            )
            map_sc = {'Normal':1, 'Partial':2, 'Abnorml':3, 'others':4}
            df['SaleCondition'] = df['SaleCondition'].map(map_sc)

    # ============== Neighborhood: target-mean encoding thủ công ==============
    if 'Neighborhood' in train_X.columns:
        nb_mean = train_df.groupby('Neighborhood')[target_col].mean()
        # map train
        train_X['Neighborhood'] = train_X['Neighborhood'].map(nb_mean)
        # test: nếu ko có trong train → fill bằng mean chung
        global_mean = train_df[target_col].mean()
        test_X['Neighborhood'] = test_X['Neighborhood'].map(nb_mean).fillna(global_mean)

        # scale
        scaler_nb = StandardScaler()
        train_X[['Neighborhood']] = scaler_nb.fit_transform(train_X[['Neighborhood']])
        test_X[['Neighborhood']] = scaler_nb.transform(test_X[['Neighborhood']])

    # ============== Feature engineering numeric ===============
    # HouseAge = YrSold - YearBuilt
    if ('YrSold' in train_X.columns) and ('YearBuilt' in train_X.columns):
        train_X['HouseAge'] = train_X['YrSold'] - train_X['YearBuilt']
        test_X['HouseAge'] = test_X['YrSold'] - test_X['YearBuilt']
        scaler_age = StandardScaler()
        train_X[['HouseAge']] = scaler_age.fit_transform(train_X[['HouseAge']])
        test_X[['HouseAge']] = scaler_age.transform(test_X[['HouseAge']])

    # IsRemodeled
    if ('YearRemodAdd' in train_X.columns) and ('YearBuilt' in train_X.columns):
        train_X['IsRemodeled'] = (train_X['YearRemodAdd'] != train_X['YearBuilt']).astype(int)
        test_X['IsRemodeled'] = (test_X['YearRemodAdd'] != test_X['YearBuilt']).astype(int)

    # Scale mấy cột liên tục quan trọng
    for col in ['TotalBsmtSF','1stFlrSF','GrLivArea','LotArea','GarageArea']:
        if col in train_X.columns:
            scaler_tmp = StandardScaler()
            train_X[[col]] = scaler_tmp.fit_transform(train_X[[col]])
            if col in test_X.columns:
                test_X[[col]] = scaler_tmp.transform(test_X[[col]])

    # Has3FullBath
    if 'FullBath' in train_X.columns:
        train_X['Has3FullBath'] = (train_X['FullBath'] >= 3).astype(int)
        test_X['Has3FullBath'] = (test_X['FullBath'] >= 3).astype(int)

    # Has3Garage
    if 'GarageCars' in train_X.columns:
        train_X['Has3Garage'] = (train_X['GarageCars'] == 3).astype(int)
        test_X['Has3Garage'] = (test_X['GarageCars'] == 3).astype(int)

    # GarageArea_per_car
    if ('GarageArea' in train_X.columns) and ('GarageCars' in train_X.columns):
        train_X['GarageArea_per_car'] = train_X['GarageArea'] / (train_X['GarageCars'] + 1)
        test_X['GarageArea_per_car'] = test_X['GarageArea'] / (test_X['GarageCars'] + 1)
        scaler_g = StandardScaler()
        train_X[['GarageArea','GarageArea_per_car']] = scaler_g.fit_transform(
            train_X[['GarageArea','GarageArea_per_car']]
        )
        test_X[['GarageArea','GarageArea_per_car']] = scaler_g.transform(
            test_X[['GarageArea','GarageArea_per_car']]
        )

    # cuối cùng: drop mấy cột gốc ko cần nữa
    drop_cols = ['YearBuilt','YrSold','YearRemodAdd','TotRmsAbvGrd']
    for df in [train_X, test_X]:
        for c in drop_cols:
            if c in df.columns:
                df.drop(columns=[c], inplace=True)

    # =============================
    # 4.2 Encode mọi object còn sót (phòng hờ)
    # =============================
    for df in [train_X, test_X]:
        obj_cols = df.select_dtypes(include='object').columns
        for col in obj_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # gắn lại target
    train_X[target_col] = train_df[target_col].values

    return train_X, test_X


train_X, test_X = prepare_features(train_clean, test_clean, target_col="SalePrice")

In [6]:
# =========================================
# 5. TÁCH X, y + ĐỒNG BỘ CỘT
# =========================================
X = train_X.drop(columns=['SalePrice'])
y_log = np.log1p(train_X['SalePrice'])

# Đồng bộ cột giữa train và test
test_X = test_X.reindex(columns=X.columns, fill_value=0)

In [19]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")  # hoặc "mean"
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
test_X = pd.DataFrame(imputer.transform(test_X), columns=test_X.columns)


In [15]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [20]:
# 6. CROSS-VALIDATION ĐỂ XEM RMSE
# =========================================
# Model 1 : XGB Stable ===================
xgb_params = dict(
    n_estimators=1800,
    learning_rate=0.02,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.55,
    min_child_weight=4,
    reg_lambda=1.0,
    reg_alpha=0.01,
    gamma=0.0,
    random_state=42,
    tree_method="hist"
)

oof_xgb = np.zeros(len(X))
pred_xgb = np.zeros(len(test_X))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y_log)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y_log.iloc[tr_idx], y_log.iloc[va_idx]

    m = XGBRegressor(**xgb_params)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    oof_xgb[va_idx] = m.predict(X_va)
    pred_xgb += m.predict(test_X) / kf.n_splits

print("XGB CV:", rmse(y_log, oof_xgb))

# ========== 7. MODEL 2: LASSO ==========
lasso = LassoCV(
    alphas=[1e-3, 3e-3, 1e-2, 3e-2, 1e-1],
    cv=5,
    random_state=42,
    max_iter=20000
)
lasso.fit(X, y_log)
oof_lasso = lasso.predict(X)
pred_lasso = lasso.predict(test_X)
print("Lasso CV (pseudo):", rmse(y_log, oof_lasso))

# ========== 8. MODEL 3: ELASTICNET ==========
enet = ElasticNetCV(
    l1_ratio=[.1, .3, .5, .7, .9, .95],
    alphas=[1e-3, 3e-3, 1e-2, 3e-2, 1e-1],
    cv=5,
    random_state=42,
    max_iter=20000
)
enet.fit(X, y_log)
oof_enet = enet.predict(X)
pred_enet = enet.predict(test_X)
print("ENet CV (pseudo):", rmse(y_log, oof_enet))

# ========== 9. BLEND ==========
# trọng số có thể thử: 0.6 / 0.25 / 0.15 hoặc 0.65 / 0.2 / 0.15
oof_blend = 0.65 * oof_xgb + 0.2 * oof_lasso + 0.15 * oof_enet
pred_blend = 0.65 * pred_xgb + 0.2 * pred_lasso + 0.15 * pred_enet

cv_blend = rmse(y_log, oof_blend)
print("BLEND CV:", cv_blend)

XGB CV: 0.14199226979781215
Lasso CV (pseudo): 0.14825330793215416
ENet CV (pseudo): 0.1490531723748843
BLEND CV: 0.13909265239440166


In [22]:
# ========== 10. SUBMISSION ==========
test_id = test["Id"]
sub = pd.DataFrame({
    "Id": test_id,
    "SalePrice": np.expm1(pred_blend)
})
sub.to_csv("submission_xgb_lasso_enet.csv", index=False)
print("✅ saved submission_xgb_lasso_enet.csv")

✅ saved submission_xgb_lasso_enet.csv


In [23]:
np.savez("prepared_data.npz", X=X, y=y_log, test_X=test_X)

import joblib

# Lưu từng model
joblib.dump(m, "xgb_model.pkl")
joblib.dump(lasso, "lasso_model.pkl")
joblib.dump(enet, "enet_model.pkl")



['enet_model.pkl']